# Search — keyword search on YouTube or Google Places, saved to Lakebase Postgres (on-demand job)

Runs the app's `search_task.py` from `src/app` (deployed with the bundle, so the job and the app share one copy of the code) on serverless. Triggered on demand (job `project1_search`: the app's "Search ingest" page or Jobs UI "Run now with different parameters"), not by the daily job.

- `mode = youtube`: YouTube search API → upsert `public.videos` / `public.channels`, append `public.search_results` (`source = search_youtube`).
- `mode = places`: Google Places Text Search (like a search on Google Maps) → upsert `public.places`, append `public.place_search_results` (`source = search_places`). Capped at 800 Text Search Enterprise calls per Vietnam month (counted in `public.search_runs`).
- Every real run writes one `public.search_runs` row.

**Secrets:** `youtube/pg_password` (Postgres role `admin`), `youtube/api_key` (YouTube Data API v3), `maps/api_key` (Places API (New)).

**Postgres driver on serverless:** serverless runs OpenSSL in FIPS mode. The OpenSSL bundled in the `psycopg[binary]` wheel fails the FIPS self-test and aborts the process (SIGABRT), so this notebook installs plain `psycopg` and sets `PSYCOPG_IMPL=python` (pure-Python implementation on the system libpq / OpenSSL).

Widget `dry_run` = `true` calls the API but writes nothing (Places: one page only, not counted).

In [ ]:
%pip install --quiet "psycopg>=3.2.0" "httpx>=0.28.0" "python-dotenv>=1.0.0"


In [ ]:
# --- Config ---
import os
import sys

dbutils.widgets.dropdown("mode", "youtube", ["youtube", "places"])
dbutils.widgets.text("keyword", "quán ăn Hồ Chí Minh")
dbutils.widgets.text("max_results", "10")
dbutils.widgets.dropdown("order", "relevance", ["relevance", "date", "viewCount"])
dbutils.widgets.text("location", "10.7769,106.7009,20000")
dbutils.widgets.text("included_type", "")
dbutils.widgets.dropdown("dry_run", "false", ["false", "true"])
dbutils.widgets.text("databricks_run_id", "")

MODE = dbutils.widgets.get("mode")
KEYWORD = dbutils.widgets.get("keyword")
MAX_RESULTS = int(dbutils.widgets.get("max_results"))
ORDER = dbutils.widgets.get("order")
LOCATION = dbutils.widgets.get("location")
INCLUDED_TYPE = dbutils.widgets.get("included_type")
DRY_RUN = dbutils.widgets.get("dry_run") == "true"
RUN_ID = dbutils.widgets.get("databricks_run_id")
RUN_ID = int(RUN_ID) if RUN_ID.isdigit() else None

os.environ.update({
    "PGHOST": "ep-still-butterfly-d8fd3r70.database.us-east-2.cloud.databricks.com",
    "PGDATABASE": "databricks_postgres",
    "PGUSER": "admin",
    "PGPORT": "5432",
    "PGSSLMODE": "require",
    "APP_SCHEMA": "public",
    "PGPASSWORD": dbutils.secrets.get("youtube", "pg_password"),
    # FIPS: use psycopg's pure-Python implementation, not the bundled OpenSSL of psycopg[binary]
    "PSYCOPG_IMPL": "python",
})
if MODE == "youtube":
    os.environ["YOUTUBE_API_KEY"] = dbutils.secrets.get("youtube", "api_key")
else:
    os.environ["MAPS_API_KEY"] = dbutils.secrets.get("maps", "api_key")

notebook_dir = os.path.dirname(dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get())
app_dir = os.path.normpath(f"/Workspace{notebook_dir}/../../app")  # src/notebooks/refresh -> src/app
sys.path.insert(0, app_dir)
print(f"App code: {app_dir} · mode = {MODE} · keyword = {KEYWORD!r} · max_results = {MAX_RESULTS} · dry_run = {DRY_RUN} · run_id = {RUN_ID}")

In [ ]:
# --- Search + save ---
import search_task

counts = search_task.run_search(
    query=KEYWORD,
    mode=MODE,
    max_results=MAX_RESULTS,
    order=ORDER,
    location=LOCATION,
    included_type=INCLUDED_TYPE,
    dry_run=DRY_RUN,
    databricks_run_id=RUN_ID,
)
print(f"✅ Search complete: { {k: v for k, v in counts.items() if k != 'top'} }")

# Return value for the Jobs API (runs/get-output → notebook_output.result), read by scripts/run_search_job.py
import json
dbutils.notebook.exit(json.dumps({"mode": MODE, "keyword": KEYWORD, "dry_run": DRY_RUN, **counts}, ensure_ascii=False, default=str))